# CatBoost RWR256 ablation — Kaggle GPU

이 노트북은 RWR 자체의 효과만 확인하는 1차 A/B 실험이다.

- A: `domain + rollup16 + gecr`
- B: `domain + rollup16 + gecr + RWR256`
- CV: canonical `fold_group5` 5-fold (`sgkf`), seed 42
- balanced sample weight, CatBoost `iterations=1000`, `learning_rate=0.05`, `depth=6`
- `cbopt10`, Pair rule, submission, ensemble은 사용하지 않는다.
- 판정값은 Pair rule 적용 전 raw OOF Macro F1의 `B - A`이다.

Kaggle Notebook 설정에서 Accelerator를 GPU로, Internet을 On으로 켠 뒤 위에서부터 실행한다.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cancer-classification-ai/onco-ai.git"
BRANCH = "feat/rwr-stacking"
REPO_DIR = Path("/kaggle/working/onco-ai")
OUTPUT_DIR = Path("/kaggle/working/cat_rwr_ablation_v1")
SEED = 42
N_SPLITS = 5
CV = "sgkf"
TAG = "cat_rwr_ablation_v1"
CONFIGS = ("dense_base", "dense_rwr256")

print("GPU environment:")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
print("Pair rule: disabled")
print("submission: disabled")

## 1. 저장소와 패키지 준비

이미 clone된 폴더는 그대로 사용한다. 새 세션이면 지정 branch를 clone한다.

In [ ]:
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "catboost==1.2.10", "scikit-learn==1.9.0",
        "numpy", "pandas", "scipy", "pyarrow", "joblib", "PyYAML",
    ],
    check=True,
)

os.chdir(REPO_DIR)
commit_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
branch_name = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
driver_text = (REPO_DIR / "scripts/train_gbdt.py").read_text(encoding="utf-8")
missing = [token for token in ("dense_base", "dense_rwr256", "--rwr-components") if token not in driver_text]
if missing:
    raise RuntimeError(f"RWR 구현이 없는 commit입니다: {missing}. branch의 최신 commit을 push했는지 확인하세요.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("branch:", branch_name)
print("commit:", commit_sha)

## 2. Kaggle Dataset 연결

Notebook의 Add Input에서 `hyunwoo11/onco-data-hack`을 추가한다. 흔한 세 가지 layout을 자동 탐색한다.

In [ ]:
def valid_raw(path: Path) -> bool:
    return all((path / name).is_file() for name in ("train.csv", "test.csv"))


def resolve_layout(root: Path) -> tuple[Path, Path | None]:
    for raw, process in (
        (root / "raw", root / "process"),
        (root / "data/raw", root / "data/process"),
        (root, root / "process"),
    ):
        if valid_raw(raw):
            return raw.resolve(), process.resolve() if process.is_dir() else None
    raise FileNotFoundError(root)


roots = [Path("/kaggle/input/onco-data-hack"), *sorted(Path("/kaggle/input").iterdir())]
layout = None
for root in roots:
    try:
        layout = resolve_layout(root)
        break
    except FileNotFoundError:
        continue
if layout is None:
    raise FileNotFoundError("Add Input에서 onco-data-hack Dataset을 연결하세요.")

RAW_SOURCE, PROCESS_SOURCE = layout
RAW_DIR = REPO_DIR / "data/raw"
PROCESS_DIR = REPO_DIR / "data/process"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESS_DIR.mkdir(parents=True, exist_ok=True)

for name in ("train.csv", "test.csv", "sample_submission.csv"):
    source = RAW_SOURCE / name
    if source.is_file():
        target = RAW_DIR / name
        if not target.exists():
            target.symlink_to(source)

if PROCESS_SOURCE is not None:
    for source in PROCESS_SOURCE.iterdir():
        if source.is_file():
            target = PROCESS_DIR / source.name
            if not target.exists():
                target.symlink_to(source)

print("raw source    :", RAW_SOURCE)
print("process source:", PROCESS_SOURCE)

## 3. 필요한 cache 준비

RWR 그래프와 SVD는 cache가 아니라 학습 중 각 fold의 train 행으로만 fit된다.

In [ ]:
FEATURES = {
    "domain_features.parquet": ["domain"],
    "sample_mutation_features_rollup.parquet": ["sample", "--include-cell-rollup"],
    "gene_event_count_matrix.parquet": ["gene", "--kind", "event_count"],
    "mutation_encoded.parquet": ["enc3"],
}

for split in ("train", "test"):
    for suffix, action in FEATURES.items():
        output = PROCESS_DIR / f"{split}_{suffix}"
        if output.is_file():
            continue
        command = [
            sys.executable, "scripts/make_features.py", action[0],
            "--split", split,
            "--input", str(RAW_DIR / f"{split}.csv"),
            "--output", str(output),
            *action[1:],
        ]
        print("RUN:", " ".join(command))
        subprocess.run(command, check=True)

fold_path = PROCESS_DIR / "train_folds.parquet"
if not fold_path.is_file():
    subprocess.run(
        [
            sys.executable, "scripts/make_folds.py",
            "--input", str(RAW_DIR / "train.csv"),
            "--out", str(fold_path),
            "--n-splits", str(N_SPLITS), "--seed", str(SEED),
        ],
        check=True,
    )

expected = [
    PROCESS_DIR / f"{split}_{suffix}"
    for split in ("train", "test") for suffix in FEATURES
] + [fold_path, fold_path.with_suffix(".json")]
missing = [str(path) for path in expected if not path.is_file()]
if missing:
    raise FileNotFoundError("누락된 cache:\n" + "\n".join(missing))
print("cache check OK:", len(expected), "files")

## 4. 고정 조건과 누수 계약 검증

조건이 하나라도 다르면 학습 전에 중단한다.

In [ ]:
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))

import catboost
import numpy as np
import pandas as pd
import sklearn

import train_gbdt as tg
from cancer_hack.class_order import CANONICAL_CLASS_ORDER

labels = pd.read_csv(RAW_DIR / "train.csv", usecols=["ID", "SUBCLASS"], dtype=str)
observed_classes = tuple(sorted(labels["SUBCLASS"].unique()))
assert observed_classes == tuple(CANONICAL_CLASS_ORDER) and len(observed_classes) == 26

folds = pd.read_parquet(fold_path)
with fold_path.with_suffix(".json").open(encoding="utf-8") as handle:
    fold_meta = json.load(handle)
assert fold_meta["seed"] == 42 and fold_meta["n_splits"] == N_SPLITS
assert "fold_group5" in fold_meta["fold_columns"] and "fold_group5" in folds.columns
assert set(folds["fold_group5"].astype(int)) == set(range(N_SPLITS))
assert folds.groupby("group_key")["fold_group5"].nunique().max() == 1
assert folds["ID"].astype(str).tolist() == labels["ID"].astype(str).tolist()

base = tg.CONFIGS["dense_base"]
rwr = tg.CONFIGS["dense_rwr256"]
assert base["blocks"] == ("domain", "rollup16", "gecr")
assert rwr["blocks"] == (*base["blocks"], "rwr256")
assert base["weight"] == rwr["weight"] == "balanced"
assert tg.MODEL_PARAMS["catboost"] == {"iterations": 1000, "learning_rate": 0.05, "depth": 6}
assert "cbopt10" not in tg.CONFIGS
assert SEED == 42 and CV == "sgkf" and N_SPLITS == 5

print("python       :", platform.python_version())
print("catboost     :", catboost.__version__)
print("scikit-learn:", sklearn.__version__)
print("classes      :", observed_classes)
print("fold sizes   :", folds["fold_group5"].value_counts().sort_index().to_dict())
print("A blocks     :", base["blocks"])
print("B blocks     :", rwr["blocks"])

## 5. CatBoost GPU A/B 실행

두 config를 같은 프로세스에서 실행한다. RWR 계산과 SVD는 CPU, CatBoost 학습은 GPU를 사용한다.

In [ ]:
command = [
    sys.executable, "scripts/train_gbdt.py",
    "--model", "catboost",
    "--configs", ",".join(CONFIGS),
    "--cv", CV,
    "--n-splits", str(N_SPLITS),
    "--folds", str(fold_path),
    "--seed", str(SEED),
    "--tag", TAG,
    "--device", "gpu",
    "--no-submission",
]
print("RUN:", " ".join(command))
subprocess.run(command, check=True)


## 6. raw OOF 결과와 판정

`delta = dense_rwr256 - dense_base`. GPU fallback 또는 파라미터 불일치도 함께 검사한다.

In [ ]:
log_dir = REPO_DIR / "artifacts/logs"
records = {}
for path in sorted(log_dir.glob(f"catboost_{TAG}_*_group5_*.json")):
    if "matrix" in path.stem or path.stem.endswith("ERROR"):
        continue
    with path.open(encoding="utf-8") as handle:
        record = json.load(handle)
    if record.get("config") in CONFIGS and record.get("seed") == SEED:
        records[record["config"]] = record

if set(records) != set(CONFIGS):
    raise RuntimeError(f"결과 로그가 부족합니다: found={sorted(records)}")

for name, record in records.items():
    assert record["cv"] == CV and record["n_splits"] == N_SPLITS
    assert record["sample_weight"] == "balanced"
    assert record["model_params"]["iterations"] == 1000
    assert record["model_params"]["learning_rate"] == 0.05
    assert record["model_params"]["depth"] == 6
    assert record["device"] == "gpu" and record["device_mixed"] is False, record["fold_models"]

assert records["dense_base"]["rwr"] is None
rwr_meta = records["dense_rwr256"]["rwr"]
assert rwr_meta["network_source"] == "fold_train_internal_comutation"
assert rwr_meta["graph_fit_scope"] == "outer_train_only"
assert rwr_meta["svd_fit_scope"] == "outer_train_only"
assert rwr_meta["validation_test_scope"] == "transform_only"
assert rwr_meta["params"]["n_components"] == 256

rows = []
for name in CONFIGS:
    record = records[name]
    rows.append({
        "config": name,
        "features": record["n_features"],
        "OOF Macro F1": record["oof_macro_f1"],
        "singleton F1": record["oof_macro_f1_singleton"],
        "device": record["device"],
        "seconds": record["elapsed_seconds"],
    })
comparison = pd.DataFrame(rows).set_index("config")
delta = records["dense_rwr256"]["oof_macro_f1"] - records["dense_base"]["oof_macro_f1"]
display(comparison)
print(f"raw OOF delta (RWR - base): {delta:+.6f}")
if delta > 0:
    print("상승: 다음 단계에서 seed 7, 2024 재현성 확인 대상입니다.")
else:
    print("상승하지 않음: 현재 판정 규칙상 RWR 탐색을 중단합니다.")

## 7. 결과 묶기

OOF, test prediction, JSON log만 보관한다. submission과 Pair rule 결과는 생성하지 않는다.

In [ ]:
for subdir in ("logs", "oof", "test_predictions"):
    source_dir = REPO_DIR / "artifacts" / subdir
    target_dir = OUTPUT_DIR / subdir
    target_dir.mkdir(parents=True, exist_ok=True)
    for source in source_dir.glob(f"*{TAG}*"):
        if source.is_file():
            shutil.copy2(source, target_dir / source.name)

manifest = {
    "branch": branch_name,
    "commit": commit_sha,
    "seed": SEED,
    "cv": CV,
    "configs": list(CONFIGS),
    "pair_rule": False,
    "submission": False,
    "raw_oof_delta_rwr_minus_base": delta,
}
with (OUTPUT_DIR / "manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, ensure_ascii=False, indent=2)

archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("saved :", OUTPUT_DIR)
print("zip   :", archive)